In [ ]:
import os
import re
import glob
import pandas as pd

os.chdir("splicing/data/rmats")

In [ ]:
max_padj = 0.00001
min_psi = .1

In [ ]:
for file in glob.glob("*/*MATS*JC.txt"):
    try:
        data = pd.read_csv(file, sep="\t", dtype=str)
    except pd.errors.EmptyDataError:
        print(f"File has no data or invalid structure: {file}")
        continue
    
    def add_quotes_to_columns(df):
        df.iloc[:, 1] = '"' + df.iloc[:, 1] + '"' 
        df.iloc[:, 2] = '"' + df.iloc[:, 2] + '"'
        return df

    def filter_PSI(data, min_psi=.1):
        return data['IncLevelDifference'].apply(lambda x: abs(float(x)) > min_psi)
        
    def filter_padj(data, max_padj=.05):
        return data['FDR'].apply(lambda x: float(x) < max_padj)

    # Filter to that inclusion level difference is > .1
    psi_mask = filter_PSI(data, min_psi)

    # Filter for max p-value
    padj_mask = filter_padj(data, max_padj)

    mask = psi_mask & padj_mask #  jc_mask & 
    filtered_data = data.loc[mask]
    
    filtered_data = add_quotes_to_columns(filtered_data)
    filename = re.sub(r"\.txt$", "", file) + f"_FDR_{max_padj}_filtered_" + file.split("/")[0] +  ".txt"
    print(filename)
    filtered_data.to_csv(filename, sep="\t", index=False, quotechar='"', quoting=3)